# Component C – Data Cleaning Pipeline
**Dataset:** UCI Online Retail (DOI 10.24432/C5BW33). Every step responds to an issue ID (Q1, Q2, ...) in Component B and is written to the audit log with before/after statistics. The raw file is read only.

**Nothing disappears silently.** Every raw row ends up in exactly one of three files: cleaned sales, returns, or excluded, each with a reason. The returns file separates product cancellations, administrative reversals and stock adjustments (column `RecordType`). A final check proves that the three add back up to the raw total.

In [1]:
# Step 0: load raw data (read only) and prepare the audit log
import os, sys
import pandas as pd

sys.path.insert(0, "..")
from quality_rules import (COUNTRY_ISO2, COUNTRY_RENAME, RECORD_ADMIN_REVERSAL, RECORD_PRODUCT_CANCELLATION,
                           RECORD_STOCK_ADJUSTMENT, admin_category, is_admin_code, is_cancellation,
                           standard_description, standard_stock_code)

raw = pd.read_excel("../data/raw/Online Retail.xlsx")
df = raw.copy()
os.makedirs("output", exist_ok=True)

audit, excluded_parts = [], []

def stats(frame):
    """The statistics recorded before and after every step."""
    return {"rows": len(frame),
            "quantity": int(frame["Quantity"].sum()),
            "revenue_gbp": round(float((frame["Quantity"] * frame["UnitPrice"]).sum()), 2),
            "invoices": int(frame["InvoiceNo"].astype(str).nunique()),
            "stock_codes": int(frame["StockCode"].astype(str).nunique()),
            "missing_customer": int(frame["CustomerID"].isna().sum()),
            "missing_description": int(frame["Description"].isna().sum())}

def log_step(step, issue, rule, reason, before, after, destination="stays in cleaned data", cells_changed=0):
    row = {"step": step, "b_issue": issue, "rule": rule, "reason": reason, "destination": destination,
           "rows_removed": before["rows"] - after["rows"], "cells_changed": cells_changed}
    for key in before:
        row[f"{key}_before"], row[f"{key}_after"] = before[key], after[key]
    audit.append(row)

def set_aside(mask, reason):
    """Move rows to the excluded file with a reason, and return the rows that stay."""
    excluded_parts.append(df[mask].assign(ExclusionReason=reason))
    return df[~mask]

print("Raw rows:", len(df))

Raw rows: 541909


### Format and type steps (no rows removed)
These run **first**, so that later rules and duplicate checks see one consistent spelling of each value.

In [2]:
# Step 1: correct data types (Q12).
before = stats(df)
numeric_id_cells = sum(int(raw[col].dropna().map(lambda v: not isinstance(v, str)).sum())
                       for col in ["InvoiceNo", "StockCode", "CustomerID"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["CustomerID"] = df["CustomerID"].astype("Int64")   # 17850.0 becomes 17850; missing stays missing
log_step("1", "Q12", "InvoiceNo, StockCode to text; CustomerID to nullable integer",
         "identifiers are labels, not measurements", before, stats(df), cells_changed=numeric_id_cells)

# Step 2: standardise StockCode case (Q9)
before = stats(df)
old = df["StockCode"].copy()
df["StockCode"] = standard_stock_code(df["StockCode"])
log_step("2", "Q9", "upper-case and trim StockCode", "85123a and 85123A are the same product",
         before, stats(df), cells_changed=int((old != df["StockCode"]).sum()))

# Step 3: standardise Description whitespace (Q10, Q11)
before = stats(df)
old = df["Description"].astype("string")
df["Description"] = standard_description(df["Description"])
log_step("3", "Q10, Q11", "trim Description and collapse repeated spaces",
         "stops one name being treated as several values; wording is not altered",
         before, stats(df), cells_changed=int((old.fillna("") != df["Description"].fillna("")).sum()))

# Step 4: standardise Country and add ISO 3166 code (Q13, FAIR gap on standard vocabularies)
before = stats(df)
old = df["Country"].copy()
df["Country"] = df["Country"].replace(COUNTRY_RENAME)
df["CountryISO2"] = df["Country"].map(COUNTRY_ISO2)
log_step("4", "Q13", "rename EIRE, RSA, USA; add CountryISO2 (blank when no single country is given)",
         "standard names and codes allow joins with external data; responds to the FAIR vocabulary gap",
         before, stats(df), cells_changed=int((old != df["Country"]).sum()))
print("Unmapped labels:", sorted(df.loc[df["CountryISO2"].isna(), "Country"].unique()))

Unmapped labels: ['Channel Islands', 'European Community', 'Unspecified']


### Row-level steps
Decision notes:
- **Duplicates:** an identical line could be a genuine second scan, and the data cannot tell. They are removed because identical invoice, product, quantity, price, minute and customer is far more likely a repeat entry. The trade-off is small: 0.22% of raw revenue.
- **Order:** cancellations and stock adjustments are separated *before* the missing-Description step, so that adjustments with no name are not lost from the returns file.
- **Types of reversal:** a `C` invoice for real goods is a *product cancellation*. A `C` invoice on an administrative code (postage, fees, discounts, manual entries) is an *administrative reversal*: a financial reversal that cannot be identified as a product return. Postage, fees and discounts are not goods; the content of manual entries is unknown. Both stay in the returns file with all original columns and values, separated by `RecordType`, so they can be analysed apart and every pound remains traceable.
- **Price 0:** this removes a few real free items, so the reason states that, and the rows are kept in the excluded file.
- **Administrative codes:** an explicit list replaces the former format test, which also deleted real products.

In [3]:
# Step 5: remove exact duplicates (Q3)
before = stats(df)
df = set_aside(df.duplicated(), "exact duplicate of an earlier row")
log_step("5", "Q3", "remove exact duplicate rows (keep first)", "identical in every field; would inflate quantity and revenue",
         before, stats(df), "excluded file")

# Step 6a: separate product cancellations (Q4)
before = stats(df)
is_cancel = is_cancellation(df["InvoiceNo"])
is_product_cancel = is_cancel & ~is_admin_code(df["StockCode"])
cancellations = df[is_product_cancel].assign(RecordType=RECORD_PRODUCT_CANCELLATION)
df = df[~is_product_cancel]
log_step("6a", "Q4", "move 'C' invoices for real products to returns file (RecordType = product cancellation)",
         "valid business events, but reversals rather than sales", before, stats(df), "returns file")

# Step 6b: separate administrative reversals (Q4, Q7)
before = stats(df)
is_admin_reversal = is_cancellation(df["InvoiceNo"])
admin_reversals = df[is_admin_reversal].assign(RecordType=RECORD_ADMIN_REVERSAL)
df = df[~is_admin_reversal]
log_step("6b", "Q4, Q7", "move 'C' invoices on administrative codes to returns file (RecordType = administrative reversal)",
         "non-product codes: postage, fees and discounts are not goods, and the content of manual entries is unknown", before, stats(df), "returns file")

# Step 7: separate stock adjustments (Q5)
before = stats(df)
is_adjustment = df["Quantity"] < 0
adjustments = df[is_adjustment].assign(RecordType=RECORD_STOCK_ADJUSTMENT)
df = df[~is_adjustment]
log_step("7", "Q5", "move Quantity < 0 without 'C' invoice to returns file",
         "all have price 0 and no customer; notes such as 'damaged' indicate warehouse corrections",
         before, stats(df), "returns file")
returns = pd.concat([cancellations, admin_reversals, adjustments]).sort_index()
returns["AdminCategory"] = admin_category(returns["StockCode"]).where(returns["RecordType"] == RECORD_ADMIN_REVERSAL)
returns["HasCustomerID"] = returns["CustomerID"].notna()
returns["LineRevenue"] = (returns["Quantity"] * returns["UnitPrice"]).round(2)

# Step 8: remove remaining rows without Description (Q2)
before = stats(df)
df = set_aside(df["Description"].isna(), "missing Description, price 0, no customer")
log_step("8", "Q2", "remove rows with missing Description", "all have price 0 and no customer: internal entries, not sales",
         before, stats(df), "excluded file")

# Step 9: remove rows with UnitPrice <= 0 (Q6)
before = stats(df)
df = set_aside(df["UnitPrice"] <= 0, "UnitPrice zero or negative")
log_step("9", "Q6", "remove rows with UnitPrice <= 0",
         "no sale value: stock notes, bad-debt entries and some free items; free items recoverable from excluded file",
         before, stats(df), "excluded file")

# Step 10: remove administrative codes (Q7)
before = stats(df)
removed_codes = df.loc[is_admin_code(df["StockCode"]), "StockCode"].value_counts()
df = set_aside(is_admin_code(df["StockCode"]), "administrative code, not a product")
log_step("10", "Q7", "remove the listed administrative StockCodes", "postage, fees, discounts and vouchers are not products",
         before, stats(df), "excluded file")
display(removed_codes)

StockCode
POST            1126
DOT              706
M                317
C2               141
BANK CHARGES      12
GIFT_0001_20       9
GIFT_0001_10       8
GIFT_0001_30       7
GIFT_0001_50       4
GIFT_0001_40       3
AMAZONFEE          2
S                  2
B                  1
Name: count, dtype: int64

### Linking sales to their cancellations
Removing `C` rows leaves the **original sale** in the data, so cancelled goods would still count as sold. Each cancellation line is therefore matched to the most recent earlier sale with the same customer, product and quantity, one-to-one. Matching is exact only: partial cancellations cannot be tied to a specific sale, so they stay unmatched and this is listed as a limitation. Only product cancellations can match a sale, because administrative codes are not in the sales data by this stage.

**Limitation:** 383 raw `C` rows (379 in the returns file, after 4 exact duplicates are removed) have no CustomerID. They cannot be linked to a customer, so any original sale they reversed cannot be flagged.

**Coverage:** the exact counts of matched and unmatched cancellations are written to `output/cleaning_residual_checks.json`. Unmatched lines may be partial cancellations, or reversals of sales before the dataset begins, or lines without a CustomerID; the data cannot separate these.

In [4]:
# Step 11: flag sales that were later cancelled in full (Q4)
before = stats(df)
open_sales = {}
columns = ["CustomerID", "StockCode", "Quantity", "InvoiceDate"]
for idx, cust, code, qty, when in df[df["CustomerID"].notna()].sort_values("InvoiceDate")[columns].itertuples():
    open_sales.setdefault((cust, code, qty), []).append((when, idx))

matched = []
for cust, code, qty, when in cancellations[cancellations["CustomerID"].notna()].sort_values("InvoiceDate")[columns].itertuples(index=False):
    earlier = [item for item in open_sales.get((cust, code, -qty), []) if item[0] <= when]
    if earlier:
        chosen = earlier[-1]
        open_sales[(cust, code, -qty)].remove(chosen)
        matched.append(chosen[1])

df["LaterCancelled"] = df.index.isin(matched)
log_step("11", "Q4", "flag sales with an exact one-to-one cancellation (LaterCancelled)",
         "lets analysts exclude reversed sales; gross revenue is otherwise overstated", before, stats(df), cells_changed=len(matched))
print("Cancellation lines:", len(cancellations), "| matched exactly:", len(matched), "| value of matched sales GBP",
      round((df.loc[matched, "Quantity"] * df.loc[matched, "UnitPrice"]).sum(), 2))

# Step 12: remove extreme reversed orders (Q14)
before = stats(df)
extreme = df["LaterCancelled"] & (df["Quantity"] >= 10000)
display(df[extreme][["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]])
df = set_aside(extreme, "extreme quantity fully reversed by a cancellation")
log_step("12", "Q14", "remove sales with Quantity >= 10,000 that were fully cancelled",
         "not real demand; other large orders are kept because many customers are wholesalers", before, stats(df), "excluded file")

# Step 13: flag missing CustomerID and add line revenue (Q1)
before = stats(df)
df = df.copy()
df["HasCustomerID"] = df["CustomerID"].notna()
df["LineRevenue"] = (df["Quantity"] * df["UnitPrice"]).round(2)
log_step("13", "Q1", "add HasCustomerID and LineRevenue (no rows removed)",
         "missingness is systematic; removing it would lose about 25% of sales and bias product analysis", before, stats(df), cells_changed=len(df))

audit_log = pd.DataFrame(audit)
pd.set_option("display.max_colwidth", 45)
audit_log[["step", "b_issue", "destination", "rows_before", "rows_after", "rows_removed", "cells_changed", "revenue_gbp_before", "revenue_gbp_after"]]

Cancellation lines: 8668 | matched exactly: 2818 | value of matched sales GBP 425796.21


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446


,step,b_issue,destination,rows_before,rows_after,rows_removed,cells_changed,revenue_gbp_before,revenue_gbp_after
0,1,Q12,stays in cleaned data,541909,541909,0,1426483,9747747.93,9747747.93
1,2,Q9,stays in cleaned data,541909,541909,0,1977,9747747.93,9747747.93
2,3,"Q10, Q11",stays in cleaned data,541909,541909,0,128367,9747747.93,9747747.93
3,4,Q13,stays in cleaned data,541909,541909,0,8545,9747747.93,9747747.93
4,5,Q3,excluded file,541909,536641,5268,0,9747747.93,9726006.95
5,6a,Q4,returns file,536641,527973,8668,0,9726006.95,10201908.11
6,6b,"Q4, Q7",returns file,527973,527390,583,0,10201908.11,10619986.68
7,7,Q5,returns file,527390,526054,1336,0,10619986.68,10619986.68
8,8,Q2,excluded file,526054,525462,592,0,10619986.68,10619986.68
9,9,Q6,excluded file,525462,524878,584,0,10619986.68,10642110.80


## Checks
These stop the notebook with an error if any rule fails, so a successful run is evidence that the pipeline holds.

In [5]:
excluded = pd.concat(excluded_parts)

# 1) Every raw row is in exactly one output
assert len(df) + len(returns) + len(excluded) == len(raw), "row reconciliation failed"
assert not (set(df.index) & set(returns.index) or set(df.index) & set(excluded.index) or set(returns.index) & set(excluded.index))

# 2) Re-run the Component B tests before and after
standard_raw_code = standard_stock_code(raw["StockCode"])
raw_text = raw["Description"].dropna().astype(str)
raw_cancel, raw_admin = is_cancellation(raw["InvoiceNo"]), is_admin_code(raw["StockCode"])

# 3) The returns file separates product and administrative reversals correctly
assert not (returns["RecordType"].eq(RECORD_PRODUCT_CANCELLATION) & is_admin_code(returns["StockCode"])).any()
assert returns.loc[returns["RecordType"].eq(RECORD_ADMIN_REVERSAL), "StockCode"].pipe(is_admin_code).all()
before_after = pd.DataFrame([
    ("Q3 duplicate rows", raw.duplicated().sum(), df[raw.columns].duplicated().sum()),
    ("Q2 missing Description", raw["Description"].isna().sum(), df["Description"].isna().sum()),
    ("Q4 'C' invoices", is_cancellation(raw["InvoiceNo"]).sum(), is_cancellation(df["InvoiceNo"]).sum()),
    ("Q5 Quantity < 0", (raw["Quantity"] < 0).sum(), (df["Quantity"] < 0).sum()),
    ("Q6 UnitPrice <= 0", (raw["UnitPrice"] <= 0).sum(), (df["UnitPrice"] <= 0).sum()),
    ("Q7 administrative code rows", is_admin_code(raw["StockCode"]).sum(), is_admin_code(df["StockCode"]).sum()),
    ("Q9 lower-case StockCode rows", (raw["StockCode"].astype(str) != standard_raw_code).sum(), (df["StockCode"] != standard_stock_code(df["StockCode"])).sum()),
    ("Q10 Description with outer spaces", (raw_text != raw_text.str.strip()).sum(), (df["Description"] != df["Description"].str.strip()).sum()),
    ("Q13 labels EIRE/RSA/USA", raw["Country"].isin(COUNTRY_RENAME).sum(), df["Country"].isin(COUNTRY_RENAME).sum()),
    ("Q14 Quantity >= 10,000", (raw["Quantity"] >= 10000).sum(), (df["Quantity"] >= 10000).sum()),
    ("Q1 missing CustomerID (kept and flagged)", raw["CustomerID"].isna().sum(), df["CustomerID"].isna().sum()),
    ("Returns file: product cancellation rows", (raw_cancel & ~raw_admin).sum(), (returns["RecordType"] == RECORD_PRODUCT_CANCELLATION).sum()),
    ("Returns file: administrative reversal rows", (raw_cancel & raw_admin).sum(), (returns["RecordType"] == RECORD_ADMIN_REVERSAL).sum()),
    ("Returns file: stock adjustment rows", (~raw_cancel & (raw["Quantity"] < 0)).sum(), (returns["RecordType"] == RECORD_STOCK_ADJUSTMENT).sum()),
    ("'C' invoice rows without CustomerID", (raw_cancel & raw["CustomerID"].isna()).sum(), (is_cancellation(returns["InvoiceNo"]) & returns["CustomerID"].isna()).sum()),
    ("Total rows", len(raw), len(df)),
    ("Quantity, sum", raw["Quantity"].sum(), df["Quantity"].sum()),
    ("Revenue GBP, sum", round((raw["Quantity"] * raw["UnitPrice"]).sum(), 2), round(df["LineRevenue"].sum(), 2)),
    ("Quantity, mean", round(raw["Quantity"].mean(), 2), round(df["Quantity"].mean(), 2)),
    ("Quantity, max", raw["Quantity"].max(), df["Quantity"].max()),
    ("UnitPrice, max", raw["UnitPrice"].max(), df["UnitPrice"].max()),
    ("Distinct StockCodes", raw["StockCode"].astype(str).nunique(), df["StockCode"].nunique()),
], columns=["measure", "before", "after"])

must_be_zero = before_after[before_after["measure"].str.match(r"Q(2|3|4|5|6|7|9|10|13|14) ")]
assert must_be_zero["after"].eq(0).all(), must_be_zero

print("Rows: raw", len(raw), "= cleaned", len(df), "+ returns", len(returns), "+ excluded", len(excluded))
print(excluded["ExclusionReason"].value_counts())

returns_summary = (returns.assign(AdminCategory=returns["AdminCategory"].fillna("(not administrative)"))
                   .groupby(["RecordType", "AdminCategory"], as_index=False)
                   .agg(rows=("Quantity", "size"), quantity=("Quantity", "sum"), value_gbp=("LineRevenue", "sum"),
                        rows_without_customer=("HasCustomerID", lambda s: int((~s).sum()))))
returns_summary["value_gbp"] = returns_summary["value_gbp"].round(2)
display(returns_summary)
before_after

Rows: raw 541909 = cleaned 522538 + returns 10587 + excluded 8784
ExclusionReason
exact duplicate of an earlier row                    5268
administrative code, not a product                   2338
missing Description, price 0, no customer             592
UnitPrice zero or negative                            584
extreme quantity fully reversed by a cancellation       2
Name: count, dtype: int64


,RecordType,AdminCategory,rows,quantity,value_gbp,rows_without_customer
0,administrative reversal,Amazon fee,32,-32,-235281.59,32
1,administrative reversal,CRUK commission,16,-16,-7933.43,0
2,administrative reversal,bank charges,25,-25,-7340.64,25
3,administrative reversal,carriage,2,-2,-65.00,1
4,administrative reversal,discount,77,-1194,-5696.22,0
5,administrative reversal,dotcom postage,1,-1,-3.29,1
6,administrative reversal,manual entry,244,-4066,-146784.46,69
7,administrative reversal,postage,126,-147,-11871.24,29
8,administrative reversal,samples,60,-60,-3102.70,60
9,product cancellation,(not administrative),8668,-270017,-475901.16,162


,measure,before,after
0,Q3 duplicate rows,5268.00,0.00
1,Q2 missing Description,1454.00,0.00
2,Q4 'C' invoices,9288.00,0.00
3,Q5 Quantity < 0,10624.00,0.00
4,Q6 UnitPrice <= 0,2517.00,0.00
5,Q7 administrative code rows,2946.00,0.00
6,Q9 lower-case StockCode rows,1977.00,0.00
7,Q10 Description with outer spaces,113452.00,0.00
8,Q13 labels EIRE/RSA/USA,8545.00,0.00
9,"Q14 Quantity >= 10,000",3.00,0.00


In [6]:
# Save outputs. The raw CSV is an unmodified export of the Excel file
raw.to_csv("output/online_retail_raw.csv", index=False, lineterminator="\n")
df.to_csv("output/online_retail_cleaned.csv", index=False, lineterminator="\n")
returns.to_csv("output/online_retail_returns.csv", index=False, lineterminator="\n")
excluded.to_csv("output/online_retail_excluded.csv", index=False, lineterminator="\n")
audit_log.to_csv("output/audit_log.csv", index=False, lineterminator="\n")
before_after.to_csv("output/before_after_summary.csv", index=False, lineterminator="\n")
returns_summary.to_csv("output/returns_summary.csv", index=False, lineterminator="\n")

# File sizes and checksums
import hashlib, json
files = ["../data/raw/Online Retail.xlsx"] + [f"output/online_retail_{name}.csv" for name in ["raw", "cleaned", "returns", "excluded"]]
manifest = [{"file": os.path.basename(f), "bytes": os.path.getsize(f),
             "sha256": hashlib.sha256(open(f, "rb").read()).hexdigest()} for f in files]
with open("output/file_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
sorted(os.listdir("output"))

['audit_log.csv',
 'before_after_summary.csv',
 'file_manifest.json',
 'online_retail_cleaned.csv',
 'online_retail_excluded.csv',
 'online_retail_raw.csv',
 'online_retail_returns.csv',
 'returns_summary.csv']

In [7]:
# Residual checks reported as limitations
is_product = returns["RecordType"] == RECORD_PRODUCT_CANCELLATION
matched_value = float((raw.loc[matched, "Quantity"] * raw.loc[matched, "UnitPrice"]).sum())
product_value = float(returns.loc[is_product, "LineRevenue"].sum())
admin = returns[returns["RecordType"] == RECORD_ADMIN_REVERSAL]
residual_rows = df[df["StockCode"].isin(["23444", "23702"])]
residual = {
    "product_cancellation_lines": int(is_product.sum()), "matched_exactly": len(matched), "unmatched": int(is_product.sum()) - len(matched),
    "matched_percent_of_lines": round(len(matched) / int(is_product.sum()) * 100, 1),
    "product_cancellation_value_gbp": round(product_value, 2), "matched_sale_value_gbp": round(matched_value, 2),
    "matched_percent_of_value": round(matched_value / -product_value * 100, 1),
    "administrative_reversal_value_gbp": round(float(admin["LineRevenue"].sum()), 2),
    "amazon_fee_value_gbp": round(float(admin.loc[admin["AdminCategory"] == "Amazon fee", "LineRevenue"].sum()), 2),
    "amazon_fee_percent_of_admin_value": round(float(admin.loc[admin["AdminCategory"] == "Amazon fee", "LineRevenue"].sum() / admin["LineRevenue"].sum() * 100), 1),
    "manual_entry_rows": int((admin["AdminCategory"] == "manual entry").sum()),
    "manual_entry_value_gbp": round(float(admin.loc[admin["AdminCategory"] == "manual entry", "LineRevenue"].sum()), 2),
    "non_product_descriptions_remaining_rows": len(residual_rows),
    "non_product_descriptions_remaining_detail": residual_rows["Description"].value_counts().to_dict(),
    "non_product_descriptions_remaining_revenue_gbp": round(float(residual_rows["LineRevenue"].sum()), 2),
    "price_zero_rows_removed_with_customer": int((excluded["ExclusionReason"].eq("UnitPrice zero or negative") & excluded["CustomerID"].notna()).sum()),
    "price_rows_removed_total": int(excluded["ExclusionReason"].eq("UnitPrice zero or negative").sum()),
}
with open("output/cleaning_residual_checks.json", "w", encoding="utf-8", newline="\n") as handle:
    json.dump(residual, handle, indent=2)
residual

{'product_cancellation_lines': 8668,
 'matched_exactly': 2818,
 'unmatched': 5850,
 'matched_percent_of_lines': 32.5,
 'product_cancellation_value_gbp': -475901.16,
 'matched_sale_value_gbp': 425796.21,
 'matched_percent_of_value': 89.5,
 'administrative_reversal_value_gbp': -418078.57,
 'amazon_fee_value_gbp': -235281.59,
 'amazon_fee_percent_of_admin_value': 56.3,
 'manual_entry_rows': 244,
 'manual_entry_value_gbp': -146784.46,
 'non_product_descriptions_remaining_rows': 82,
 'non_product_descriptions_remaining_detail': {'Next Day Carriage': 79,
  'High Resolution Image': 3},
 'non_product_descriptions_remaining_revenue_gbp': 1227.0,
 'price_zero_rows_removed_with_customer': 40,
 'price_rows_removed_total': 584}